# DINOv2 + CNN Refiner — RSNA 2026 Knee Abnormality Detection

### Architecture
```
5-slice MRI stack → DINOv2 ViT-S (frozen, per-slice)  ─┐
                      → CNN SPA (trainable, spatial)   ─┤→ CrossModalFusion → SliceTransformer → Head → 12 logits
```

### Data
- **Train**: ~2,000 studies with HIGH-confidence pseudo-labels (from LLM)
- **Val**: 58 gold-labeled studies (competition ground truth)
- **Input**: 5 adjacent Sagittal T2/PD FS slices, 392×392 (28×14 patches)
- **Cache**: All DICOM volumes pre-loaded into RAM (~12-18 GB) for instant access

### Hardware
- **T4 ×2 with DataParallel** for 9-hour Kaggle sessions

## 1. Setup & Install Dependencies

In [ ]:
!pip install -q timm pydicom opencv-python scikit-learn

## 2. Imports & Configuration

In [ ]:
from __future__ import annotations

import gc, math, os, sys, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
import pydicom
import cv2
from sklearn.metrics import roc_auc_score

print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda}')
print(f'GPU count: {torch.cuda.device_count()}')

In [ ]:
# ============================================================
# Configuration — adjust these for your experiment
# ============================================================

TARGET_COLUMNS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA',
    'Effusion', 'Synovitis', "Baker's",
    'Contusion', 'Fracture',
]

CFG = {
    # --- Paths (Kaggle) ---
    'comp_input':   '/kaggle/input/competitions/rsna-knee-abnormality-detection',
    'pseudo_input': '/kaggle/input/datasets/easoncyy/rsna-knee-pseudo-labels',
    'dicom_subdir': 'train_series',
    'output_dir':   '/kaggle/working',

    # --- Data ---
    'image_size': 392,          # MUST be divisible by 14: 392=28×14 patches
    'slice_count': 5,
    'center_stride': 3,         # sample every Nth center slice (1=all, 3=~1/3 samples)
    'confidence_filter': 'HIGH',

    # --- Model ---
    'dinov2_variant': 'vit_small_patch14_dinov2.lvd142m',
    'cls_dim': 384,
    'spa_channels': 64,
    'num_classes': 12,

    # --- Training ---
    'batch_size': 32,           # total batch size (DataParallel splits across GPUs)
    'epochs': 50,
    'lr': 2e-4,
    'weight_decay': 1e-4,
    'lr_t0': 10,
    'lr_t_mult': 2,
    'lr_eta_min': 1e-6,
    'focal_gamma': 2.0,
    'focal_alpha': 0.25,
    'dropout': 0.1,
    'head_dropout': 0.3,
    'grad_clip': 1.0,
    'early_stop_patience': 10,
    'mixed_precision': True,
    'num_workers': 4,           # RAM cache makes I/O fast; 4 workers is enough
}

# Device setup
N_GPUS = torch.cuda.device_count()
DEVICE = torch.device('cuda' if N_GPUS > 0 else 'cpu')
IS_MAIN = True  # single-process with DataParallel

if IS_MAIN:
    print(f'GPUs: {N_GPUS} | Device: {DEVICE}')
    for k, v in CFG.items():
        print(f'  {k}: {v}')

## 3. Model Components

In [ ]:
# ============================================================
# 3a. CNN Spatial Pattern Adapter (SPA)
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.GELU()
    def forward(self, x): return self.act(self.bn(self.conv(x)))


class SPAModule(nn.Module):
    """Multi-scale spatial feature extractor. ~100K params."""
    def __init__(self, in_channels=5, base_ch=64):
        super().__init__()
        self.stem = ConvBlock(in_channels, base_ch)
        self.stage1 = nn.Sequential(ConvBlock(base_ch, base_ch), ConvBlock(base_ch, base_ch, 2))
        self.stage2 = nn.Sequential(ConvBlock(base_ch, base_ch*2), ConvBlock(base_ch*2, base_ch*2, 2))
        self.stage3 = nn.Sequential(ConvBlock(base_ch*2, base_ch*4), ConvBlock(base_ch*4, base_ch*4, 2))

    def forward(self, x):
        x = self.stem(x)       # [B, 64,  392, 392]
        s2 = self.stage1(x)    # [B, 64,  196, 196]
        s4 = self.stage2(s2)   # [B, 128, 98,  98]
        s8 = self.stage3(s4)   # [B, 256, 49,  49]
        return {'s2': s2, 's4': s4, 's8': s8}

In [ ]:
# ============================================================
# 3b. Cross-Modal Fusion (LGFA-style)
# ============================================================

class CrossModalFusion(nn.Module):
    """Cross-attention: [CLS] token queries CNN spatial features."""
    def __init__(self, cls_dim=384, cnn_dim=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = cls_dim // num_heads
        self.cnn_proj = nn.Linear(cnn_dim, cls_dim)
        self.q_proj = nn.Linear(cls_dim, cls_dim)
        self.k_proj = nn.Linear(cls_dim, cls_dim)
        self.v_proj = nn.Linear(cls_dim, cls_dim)
        self.out_proj = nn.Linear(cls_dim, cls_dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(cls_dim)
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, cls_token, cnn_features):
        B, C, H, W = cnn_features.shape
        D = cls_token.shape[-1]
        cnn_seq = cnn_features.flatten(2).transpose(1, 2)
        cnn_seq = self.cnn_proj(cnn_seq)

        q = self.q_proj(cls_token).view(B, 1, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(cnn_seq).view(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(cnn_seq).view(B, -1, self.num_heads, self.head_dim).transpose(1, 2)

        scale = self.head_dim ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale
        attn = self.dropout(attn.softmax(dim=-1))
        out = (attn @ v).transpose(1, 2).contiguous().view(B, D)
        out = self.out_proj(out)

        gate = self.gate.tanh()
        return self.norm(cls_token + gate * out)

In [ ]:
# ============================================================
# 3c. Slice Transformer
# ============================================================

class SliceTransformer(nn.Module):
    """Self-attention over 5 adjacent slice features."""
    def __init__(self, dim=384, num_heads=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(1, 5, dim) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=num_heads, dim_feedforward=dim*4,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.norm = nn.LayerNorm(dim)

    def forward(self, slice_features):
        tokens = slice_features + self.pos_embed
        tokens = self.transformer(tokens)
        return self.norm(tokens.mean(dim=1))

In [ ]:
# ============================================================
# 3d. Classification Head
# ============================================================

class ClassificationHead(nn.Module):
    """Linear→LayerNorm→GELU→Dropout→Linear→12 logits"""
    def __init__(self, in_features=384, hidden=512, num_classes=12, dropout=0.3):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )
    def forward(self, x): return self.head(x)

In [ ]:
# ============================================================
# 3e. Full DINOv2Refiner Model
# ============================================================

class DINOv2Refiner(nn.Module):
    """DINOv2 (frozen) + SPA + CrossModalFusion + SliceTransformer + Head."""
    def __init__(self, dinov2_model, spa_channels=64, cls_dim=384,
                 num_slices=5, num_classes=12, num_heads=4,
                 st_layers=2, dropout=0.1, freeze_dinov2=True):
        super().__init__()
        self.num_slices = num_slices
        self.cls_dim = cls_dim

        self.dinov2 = dinov2_model
        if freeze_dinov2:
            for p in self.dinov2.parameters():
                p.requires_grad = False
            self.dinov2.eval()

        self.spa = SPAModule(in_channels=num_slices, base_ch=spa_channels)
        self.fusion = CrossModalFusion(cls_dim=cls_dim, cnn_dim=spa_channels*4,
                                       num_heads=num_heads, dropout=dropout)
        self.slice_transformer = SliceTransformer(dim=cls_dim, num_heads=num_heads,
                                                   num_layers=st_layers, dropout=dropout)
        self.head = ClassificationHead(in_features=cls_dim, hidden=512,
                                       num_classes=num_classes, dropout=CFG['head_dropout'])

        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        if IS_MAIN:
            print(f'[DINOv2Refiner] Total: {total:,} | Trainable: {trainable:,} '
                  f'({trainable/total*100:.0f}%) | Frozen: {total-trainable:,}')

    def _extract_cls_batched(self, x_3ch):
        """Extract [CLS] tokens from batched 3-channel inputs. x_3ch: [B_total, 3, H, W]"""
        with torch.no_grad():
            features = self.dinov2.forward_features(x_3ch)
        return features[:, 0, :]  # [B_total, cls_dim]

    def forward(self, x):
        B = x.shape[0]

        # ── SPA on 5-channel input ─────────────────────────────
        spa_features = self.spa(x)  # s8: [B, 256, 49, 49]

        # ── Batched DINOv2: [B, 5, H, W] → [B*5, 3, H, W] ────
        # 1) Stack slices into batch dimension:
        #    permute to [5, B, H, W] → reshape to [B*5, 1, H, W] → expand to 3-ch
        #    Order after reshape: slice0_b0, slice0_b1, ..., slice0_bB-1, slice1_b0, ...
        x_5bhw = x.permute(1, 0, 2, 3).contiguous()        # [5, B, H, W]
        x_flat = x_5bhw.view(B * self.num_slices, 1, x.shape[-2], x.shape[-1])  # [B*5, 1, H, W]
        x_flat_3ch = x_flat.expand(-1, 3, -1, -1)          # [B*5, 3, H, W]
        all_cls = self._extract_cls_batched(x_flat_3ch)     # [B*5, cls_dim]

        # 2) Reshape back to [B, 5, cls_dim]:
        #    [5, B, cls_dim] → transpose → [B, 5, cls_dim]
        all_cls = all_cls.view(self.num_slices, B, self.cls_dim)  # [5, B, cls_dim]
        all_cls = all_cls.transpose(0, 1).contiguous()            # [B, 5, cls_dim]

        # ── Cross-modal fusion per slice ──────────────────────
        fused = []
        for i in range(self.num_slices):
            enhanced = self.fusion(all_cls[:, i, :], spa_features['s8'])
            fused.append(enhanced)

        slice_tokens = torch.stack(fused, dim=1)            # [B, 5, cls_dim]
        study_feature = self.slice_transformer(slice_tokens)
        return self.head(study_feature)

    def train(self, mode=True):
        super().train(mode)
        self.dinov2.eval()
        return self

In [ ]:
# ============================================================
# 3f. Focal BCE Loss
# ============================================================

class FocalBCELoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        p_t = targets * probs + (1 - targets) * (1 - probs)
        focal_weight = (1.0 - p_t) ** self.gamma
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        return (alpha_weight * focal_weight * bce).mean()

## 4. DICOM Dataset

In [ ]:
# ============================================================
# DICOM I/O helpers (fast path: specific_tags to minimise I/O)
# ============================================================

PLANE_SORT_AXIS = {'Sagittal': 0, 'Coronal': 1, 'Axial': 2}

# pydicom specific_tags — only parse what we actually need
# CRITICAL: must include ImagePixelModule tags (Rows, Columns, BitsAllocated,
# SamplesPerPixel, PhotometricInterpretation, PixelRepresentation) because
# ds.pixel_array requires them. Without them, pixel_array raises AttributeError
# on every slice → "No DICOM readable" on every series.
_DICOM_SPECIFIC_TAGS = [
    # --- Sorting / position ---
    (0x0020, 0x0032),  # ImagePositionPatient
    (0x0020, 0x1041),  # SliceLocation
    (0x0020, 0x0013),  # InstanceNumber
    # --- Pixel decoding (needed by ds.pixel_array) ---
    (0x0028, 0x0002),  # SamplesPerPixel
    (0x0028, 0x0004),  # PhotometricInterpretation
    (0x0028, 0x0010),  # Rows
    (0x0028, 0x0011),  # Columns
    (0x0028, 0x0100),  # BitsAllocated
    (0x0028, 0x0101),  # BitsStored
    (0x0028, 0x0102),  # HighBit
    (0x0028, 0x0103),  # PixelRepresentation
    # --- Pixel data itself ---
    (0x7FE0, 0x0010),  # PixelData
]

def _get_slice_position(ds, plane=None):
    try:
        ipp = getattr(ds, 'ImagePositionPatient', None)
        if ipp and len(ipp) >= 3:
            axis = PLANE_SORT_AXIS.get(plane, 2) if plane else 2
            return float(ipp[axis])
    except: pass
    try:
        sl = getattr(ds, 'SliceLocation', None)
        if sl is not None: return float(sl)
    except: pass
    try: return float(getattr(ds, 'InstanceNumber', 0))
    except: return 0.0


def read_dicom_series(series_dir, plane=None, image_size=392, lower_pct=0.5, upper_pct=99.5):
    series_dir = Path(series_dir)
    dcm_paths = sorted(series_dir.glob('*.dcm'))
    if not dcm_paths: dcm_paths = sorted(series_dir.glob('*'))

    slices_info = []
    for p in dcm_paths:
        try:
            # specific_tags: skip 90%+ of DICOM metadata parsing
            # (includes all ImagePixelModule tags needed by pixel_array)
            ds = pydicom.dcmread(str(p), force=True, specific_tags=_DICOM_SPECIFIC_TAGS)
            pos = _get_slice_position(ds, plane)
            img = ds.pixel_array.astype(np.float32)
            slices_info.append((pos, img))
        except: continue

    if not slices_info: raise RuntimeError(f'No DICOM readable: {series_dir}')
    slices_info.sort(key=lambda x: x[0])
    images = np.stack([img for _, img in slices_info], axis=0)

    v_low = np.percentile(images, lower_pct)
    v_high = np.percentile(images, upper_pct)
    images = np.clip(images, v_low, v_high)
    images = (images - v_low) / max(v_high - v_low, 1e-6)

    resized = []
    for img in images:
        r = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
        resized.append(r)
    return np.stack(resized, axis=0).astype(np.float32)

In [ ]:
# ============================================================
# 2.5D Dataset — supports RAM volume cache for fast loading
# ============================================================

class Knee25DPseudoDataset(Dataset):
    def __init__(self, series_df, labels_df, dicom_root, image_size=392,
                 slice_count=5, is_train=True, center_stride=3,
                 volume_cache=None):
        self.dicom_root = Path(dicom_root)
        self.image_size = image_size
        self.slice_count = slice_count
        self.is_train = is_train
        self.half_window = slice_count // 2
        self.volume_cache = volume_cache  # dict: {dicom_dir_str: np.ndarray} or None

        df = series_df.copy()
        df = df[df['Anatomical_Plane'] == 'Sagittal']
        if 'Fluid_Sensitive' in df.columns: df = df[df['Fluid_Sensitive'] == 1]
        if 'Fat_Suppression' in df.columns: df = df[df['Fat_Suppression'] == 1]

        self.label_map = labels_df[TARGET_COLUMNS].apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float32)

        self.samples = []
        skipped = 0
        for (study_uid, series_uid), grp in df.groupby(['StudyInstanceUID', 'SeriesInstanceUID']):
            if study_uid not in self.label_map.index:
                skipped += 1; continue
            plane = grp.iloc[0]['Anatomical_Plane']
            dicom_dir = self.dicom_root / study_uid / series_uid
            dicom_dir_str = str(dicom_dir)

            # When cache is provided, check it exists in cache
            if self.volume_cache is not None and dicom_dir_str not in self.volume_cache:
                skipped += 1; continue

            # When no cache, check DICOM dir exists on disk
            if self.volume_cache is None:
                if not dicom_dir.exists():
                    skipped += 1; continue
                dcm_files = list(dicom_dir.glob('*.dcm'))
                if not dcm_files: dcm_files = list(dicom_dir.glob('*'))
                n_slices = len(dcm_files)
            else:
                n_slices = self.volume_cache[dicom_dir_str].shape[0]

            if n_slices < 3:
                skipped += 1; continue

            labels = self.label_map.loc[study_uid].values.astype(np.float32)
            for center_idx in range(0, n_slices, center_stride):
                self.samples.append({'study_uid': study_uid, 'series_uid': series_uid,
                                     'dicom_dir': dicom_dir_str, 'plane': plane,
                                     'center_idx': center_idx, 'n_slices': n_slices,
                                     'labels': labels})
        if IS_MAIN and skipped:
            print(f'[{type(self).__name__}] {skipped} series skipped')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Fast path: read from RAM cache
        if self.volume_cache is not None:
            volume = self.volume_cache[sample['dicom_dir']]
        else:
            # Slow path: read DICOM from disk (fallback)
            try:
                volume = read_dicom_series(sample['dicom_dir'], plane=sample['plane'],
                                           image_size=self.image_size)
            except:
                return {'image': torch.zeros(self.slice_count, self.image_size, self.image_size),
                        'labels': torch.from_numpy(sample['labels']),
                        'study_uid': sample['study_uid'], 'plane': sample['plane']}

        center = sample['center_idx']
        n_total = volume.shape[0]
        half = self.half_window
        indices = [max(0, min(n_total-1, center+o)) for o in range(-half, half+1)]
        stack = volume[indices]
        return {'image': torch.from_numpy(stack.copy()),
                'labels': torch.from_numpy(sample['labels']),
                'study_uid': sample['study_uid'], 'plane': sample['plane']}

## 5. Load & Merge Data

In [ ]:
comp_input = Path(CFG['comp_input'])
pseudo_input = Path(CFG['pseudo_input'])

# Load competition metadata
train_meta = pd.read_csv(comp_input / 'train.csv')
series_meta = pd.read_csv(comp_input / 'train_series.csv')

# Split gold vs unlabeled
label_cols_present = [c for c in TARGET_COLUMNS if c in train_meta.columns]
has_gold = train_meta[label_cols_present].notna().all(axis=1)
gold_df = train_meta[has_gold].copy()

if IS_MAIN:
    print(f'Gold studies: {len(gold_df)}')
    print(f'Total studies: {len(train_meta)}')

# Load pseudo-labels
pseudo_df = pd.read_csv(pseudo_input / 'pseudo_labels.csv')

# Filter by confidence
if CFG['confidence_filter'] == 'HIGH':
    conf_mask = pd.Series(True, index=pseudo_df.index)
    for c in TARGET_COLUMNS:
        conf_mask &= (pseudo_df[f'conf_{c}'] == 'HIGH')
    pseudo_df = pseudo_df[conf_mask]
elif CFG['confidence_filter'] == 'HIGH_PLUS_MEDIUM':
    conf_mask = pd.Series(True, index=pseudo_df.index)
    for c in TARGET_COLUMNS:
        conf_mask &= (pseudo_df[f'conf_{c}'].isin(['HIGH', 'MEDIUM']))
    pseudo_df = pseudo_df[conf_mask]

# Build label DataFrames
train_labels = pseudo_df[['StudyInstanceUID']].copy()
for c in TARGET_COLUMNS:
    train_labels[c] = pseudo_df[f'pred_{c}']
train_labels = train_labels.set_index('StudyInstanceUID')
train_labels = train_labels.apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float32)

val_labels = gold_df[['StudyInstanceUID'] + label_cols_present].copy()
val_labels = val_labels.set_index('StudyInstanceUID')
for c in TARGET_COLUMNS:
    if c not in val_labels.columns:
        val_labels[c] = 0.0
val_labels = val_labels.apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float32)

if IS_MAIN:
    print(f'Train studies (pseudo): {len(train_labels)}')
    print(f'Val studies (gold):    {len(val_labels)}')
    for c in TARGET_COLUMNS:
        t_pos = (train_labels[c] == 1).sum()
        v_pos = (val_labels[c] == 1).sum()
        print(f'  {c:<20s}  pseudo_pos={t_pos:5d}  gold_pos={v_pos:3d}')

In [ ]:
# ============================================================
# Build RAM volume cache — pre-load all DICOM series once
# This is the KEY optimisation: subsequent epochs read from RAM (instant)
# instead of Kaggle's slow network filesystem (~500ms per series)
# ============================================================

dicom_root = Path(CFG['comp_input']) / CFG['dicom_subdir']
print(f'DICOM root: {dicom_root}')
print(f'DICOM root exists: {dicom_root.exists()}')

# Collect all unique series directories needed for train + val
def _collect_series_dirs(series_df, labels_df):
    """Return set of unique DICOM series dirs for Sagittal T2 FS series with labels."""
    df = series_df.copy()
    df = df[df['Anatomical_Plane'] == 'Sagittal']
    if 'Fluid_Sensitive' in df.columns: df = df[df['Fluid_Sensitive'] == 1]
    if 'Fat_Suppression' in df.columns: df = df[df['Fat_Suppression'] == 1]

    dirs = set()
    for (study_uid, series_uid), grp in df.groupby(['StudyInstanceUID', 'SeriesInstanceUID']):
        if study_uid not in labels_df.index:
            continue
        d = dicom_root / study_uid / series_uid
        if d.exists():
            dirs.add(str(d))
    return dirs

train_dirs = _collect_series_dirs(series_meta, train_labels)
val_dirs = _collect_series_dirs(series_meta, val_labels)
all_dirs = train_dirs | val_dirs

print(f'\nUnique series to cache: {len(all_dirs)}  (train: {len(train_dirs)}, val: {len(val_dirs)})')

# ── Read all volumes into RAM ────────────────────────────────
VOLUME_CACHE = {}  # {dicom_dir_str: np.ndarray [N, H, W] float16}
t_cache = time.time()
failed_series = []  # track failures for summary

for i, d in enumerate(sorted(all_dirs)):
    try:
        vol = read_dicom_series(d, plane='Sagittal', image_size=CFG['image_size'])
        VOLUME_CACHE[d] = vol.astype(np.float16)  # float16: 2 bytes/pixel, saves ~50% RAM
    except Exception as e:
        failed_series.append((d, str(e)))
        continue

    if (i + 1) % 500 == 0:
        elapsed = time.time() - t_cache
        ram_gb = sum(v.nbytes for v in VOLUME_CACHE.values()) / 1024**3
        print(f'  Cached {i+1}/{len(all_dirs)} series | {ram_gb:.1f} GB RAM | {elapsed:.0f}s')

cache_time = time.time() - t_cache
ram_gb = sum(v.nbytes for v in VOLUME_CACHE.values()) / 1024**3
print(f'\n✓ Volume cache complete: {len(VOLUME_CACHE)} series, {ram_gb:.1f} GB in {cache_time:.0f}s')

# ── Failure summary ──────────────────────────────────────────
if failed_series:
    print(f'\n⚠️  {len(failed_series)}/{len(all_dirs)} series failed to load:')
    for d, err in failed_series[:8]:  # print first 8 with details
        print(f'    - {Path(d).parent.name[:40]}.../...{Path(d).name[:40]}: {err[:80]}')
    if len(failed_series) > 8:
        print(f'    ... and {len(failed_series)-8} more')
    print(f'  These will be skipped by the dataset (n_slices < 3 check).')
    print(f'  Impact: {len(failed_series)/len(all_dirs)*100:.1f}% of series missing — '
          f'{"negligible" if len(failed_series)/len(all_dirs) < 0.02 else "notable, consider investigating"}')
else:
    print(f'  All series loaded successfully!')

# Force garbage collection before creating datasets
gc.collect()

In [ ]:
# Create datasets (using RAM cache — __getitem__ will be near-instant)
print(f'\n--- Creating datasets ---')

train_ds = Knee25DPseudoDataset(series_meta, train_labels, dicom_root,
                                 image_size=CFG['image_size'], slice_count=CFG['slice_count'],
                                 is_train=True, center_stride=CFG['center_stride'],
                                 volume_cache=VOLUME_CACHE)
val_ds = Knee25DPseudoDataset(series_meta, val_labels, dicom_root,
                               image_size=CFG['image_size'], slice_count=CFG['slice_count'],
                               is_train=False, center_stride=1,  # val: all positions
                               volume_cache=VOLUME_CACHE)

if IS_MAIN:
    print(f'Train samples: {len(train_ds):,}  (stride={CFG["center_stride"]})')
    print(f'Val samples:   {len(val_ds):,}  (stride=1, all positions)')
    if len(train_ds) == 0:
        print('⚠️  Train dataset is EMPTY!')
    if len(val_ds) == 0:
        print('⚠️  Val dataset is EMPTY!')

## 6. Build Model

In [ ]:
# DataLoader — persistent workers keep the cache warm across epochs
loader_kw = dict(num_workers=CFG['num_workers'], pin_memory=True, prefetch_factor=2,
                 persistent_workers=True if CFG['num_workers'] > 0 else False)
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True, **loader_kw)
val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False, **loader_kw)

if IS_MAIN:
    print(f'Train batches: {len(train_loader):,}  (batch_size={CFG["batch_size"]})')
    print(f'Val batches:   {len(val_loader):,}')

In [ ]:
if IS_MAIN: print('Loading DINOv2 backbone...')

dinov2_backbone = timm.create_model(
    CFG['dinov2_variant'], pretrained=True, num_classes=0,
    img_size=CFG['image_size'],
)

model = DINOv2Refiner(
    dinov2_model=dinov2_backbone,
    spa_channels=CFG['spa_channels'],
    cls_dim=CFG['cls_dim'],
    num_slices=CFG['slice_count'],
    num_classes=CFG['num_classes'],
    num_heads=4, st_layers=2,
    dropout=CFG['dropout'],
    freeze_dinov2=True,
).to(DEVICE)

# DataParallel: automatically splits batch across both T4 GPUs
if N_GPUS > 1:
    model = nn.DataParallel(model)
    print(f'[Model] Wrapped with DataParallel across {N_GPUS} GPUs')

criterion = FocalBCELoss(gamma=CFG['focal_gamma'], alpha=CFG['focal_alpha'])

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=CFG['lr_t0'], T_mult=CFG['lr_t_mult'], eta_min=CFG['lr_eta_min'])

scaler = torch.amp.GradScaler('cuda') if CFG['mixed_precision'] else None

In [ ]:
def train_epoch(model, loader, optimizer, criterion, scaler, epoch):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    use_amp = scaler is not None

    for bi, batch in enumerate(loader):
        images = batch['image'].to(DEVICE, non_blocking=True)
        labels = batch['labels'].to(DEVICE, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)

        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()

        if IS_MAIN and bi % 20 == 0:
            print(f'  Epoch {epoch:3d} [{bi:4d}/{len(loader):4d}] loss={loss.item():.4f}', flush=True)

    return total_loss / len(loader)


@torch.no_grad()
def validate_epoch(model, loader, criterion):
    """Study-level validation: aggregate slice predictions per study, then score."""
    model.eval()

    study_probs = defaultdict(list)
    study_targets_dict = {}
    total_loss = 0.0
    n_batches = 0

    for batch in loader:
        images = batch['image'].to(DEVICE, non_blocking=True)
        labels = batch['labels'].to(DEVICE, non_blocking=True)
        uids = batch['study_uid']

        logits = model(images)
        total_loss += criterion(logits, labels).item()
        n_batches += 1

        probs = torch.sigmoid(logits).cpu().numpy()

        for i, uid in enumerate(uids):
            study_probs[uid].append(probs[i])
            if uid not in study_targets_dict:
                study_targets_dict[uid] = labels[i].cpu().numpy()

    study_uids = list(study_targets_dict.keys())
    study_preds = np.zeros((len(study_uids), 12), dtype=np.float32)
    study_targets = np.zeros((len(study_uids), 12), dtype=np.float32)

    for i, uid in enumerate(study_uids):
        study_preds[i] = np.mean(study_probs[uid], axis=0)
        study_targets[i] = study_targets_dict[uid]

    per_class = {}
    aucs = []

    for i, c in enumerate(TARGET_COLUMNS):
        y_true = study_targets[:, i]
        y_prob = study_preds[:, i]
        n_pos = int(y_true.sum())

        metrics = {'auc': float('nan'), 'accuracy': float('nan'),
                   'precision': float('nan'), 'recall': float('nan'),
                   'f1': float('nan'), 'n_pos': n_pos, 'n_total': len(y_true)}

        if n_pos == 0 or n_pos == len(y_true):
            y_pred_binary = (y_prob >= 0.5).astype(int)
            metrics['accuracy'] = float((y_true == y_pred_binary).mean())
            per_class[c] = metrics
            continue

        try:
            a = roc_auc_score(y_true, y_prob)
            metrics['auc'] = float(a)
            aucs.append(a)
        except Exception:
            pass

        y_pred_binary = (y_prob >= 0.5).astype(int)
        tp = int(((y_pred_binary == 1) & (y_true == 1)).sum())
        fp = int(((y_pred_binary == 1) & (y_true == 0)).sum())
        fn = int(((y_pred_binary == 0) & (y_true == 1)).sum())
        tn = int(((y_pred_binary == 0) & (y_true == 0)).sum())

        metrics['accuracy'] = float((tp + tn) / len(y_true))
        metrics['precision'] = float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0
        metrics['recall'] = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
        metrics['f1'] = float(2 * metrics['precision'] * metrics['recall'] /
                              (metrics['precision'] + metrics['recall'])) if (metrics['precision'] + metrics['recall']) > 0 else 0.0
        per_class[c] = metrics

    return {
        'loss': total_loss / max(n_batches, 1),
        'macro_auc': float(np.mean(aucs)) if aucs else 0.0,
        'per_class': per_class,
        'study_preds': study_preds,
        'study_targets': study_targets,
        'study_uids': study_uids,
    }


def save_validation_report(val_metrics, output_dir, epoch=None, is_best=False):
    """Save per-class metrics CSV + raw study predictions CSV."""
    out = Path(output_dir)

    rows = []
    for c in TARGET_COLUMNS:
        m = val_metrics['per_class'][c]
        rows.append({'class': c, 'auc': m['auc'], 'accuracy': m['accuracy'],
                     'precision': m['precision'], 'recall': m['recall'],
                     'f1': m['f1'], 'n_pos': m['n_pos'], 'n_total': m['n_total']})

    report_df = pd.DataFrame(rows)
    report_df['macro_auc'] = val_metrics['macro_auc']
    report_df['val_loss'] = val_metrics['loss']

    tag = f'_epoch{epoch}' if epoch else ''
    if is_best:
        tag = '_best'

    report_path = out / f'validation_report{tag}.csv'
    report_df.to_csv(report_path, index=False)
    print(f'  Validation report saved: {report_path}')

    study_rows = []
    for i, uid in enumerate(val_metrics['study_uids']):
        row = {'StudyInstanceUID': uid}
        for j, c in enumerate(TARGET_COLUMNS):
            row[f'true_{c}'] = int(val_metrics['study_targets'][i, j])
            row[f'pred_{c}'] = float(val_metrics['study_preds'][i, j])
        study_rows.append(row)

    preds_df = pd.DataFrame(study_rows)
    preds_path = out / f'validation_predictions{tag}.csv'
    preds_df.to_csv(preds_path, index=False)
    print(f'  Study predictions saved: {preds_path} ({len(study_rows)} studies)')

    return report_df


def print_validation_summary(val_metrics):
    """Print a formatted table of per-class results."""
    print(f'\n  {"Class":<20s} {"AUC":>7s} {"Acc":>7s} {"Prec":>7s} {"Rec":>7s} {"F1":>7s} {"Pos":>5s}')
    print(f'  {"-"*20} {"-"*7} {"-"*7} {"-"*7} {"-"*7} {"-"*7} {"-"*5}')

    for c in TARGET_COLUMNS:
        m = val_metrics['per_class'][c]
        auc_str = f'{m["auc"]:.3f}' if not math.isnan(m['auc']) else '  N/A  '
        print(f'  {c:<20s} {auc_str:>7s} {m["accuracy"]:7.3f} {m["precision"]:7.3f} '
              f'{m["recall"]:7.3f} {m["f1"]:7.3f} {m["n_pos"]:5d}')

    print(f'  {"-"*20} {"-"*7} {"-"*7} {"-"*7} {"-"*7} {"-"*7} {"-"*5}')
    print(f'  {"Macro AUC":<20s} {val_metrics["macro_auc"]:7.3f}')
    print()

    few_pos = [c for c in TARGET_COLUMNS if val_metrics['per_class'][c]['n_pos'] < 5]
    if few_pos:
        print(f'  ⚠️  Classes with <5 positive samples (metrics unreliable):')
        for c in few_pos:
            n = val_metrics['per_class'][c]['n_pos']
            print(f'      {c}: n_pos={n}')

## 7. Training Loop

## 8. Quick Sanity Check (optional)

Run a single batch to verify the pipeline works before committing to full training.

In [ ]:
# Sanity check — run one batch and print shapes
if IS_MAIN:
    batch = next(iter(train_loader))
    print(f'Image shape:  {batch["image"].shape}')   # [B, 5, 392, 392]
    print(f'Label shape:  {batch["labels"].shape}')  # [B, 12]
    print(f'Study UIDs:   {batch["study_uid"][:3]}')

    with torch.no_grad():
        out = model(batch['image'].to(DEVICE))
    print(f'Output shape: {out.shape}')  # [B, 12]
    print(f'Output range: [{out.min().item():.3f}, {out.max().item():.3f}]')
    if N_GPUS > 1:
        print(f'DataParallel: batch split across {N_GPUS} GPUs')

In [ ]:
if IS_MAIN:
    print(f'\n{"="*60}')
    print(f'Starting training — {CFG["epochs"]} epochs | Batch={CFG["batch_size"]} (×{N_GPUS} GPUs via DataParallel)')
    print(f'DINOv2: {CFG["dinov2_variant"]} | Confidence: {CFG["confidence_filter"]}')
    print(f'Train samples: {len(train_ds):,} | Val samples: {len(val_ds):,}')
    print(f'{"="*60}\n')

best_auc = 0.0
best_epoch = 0
patience = 0
ckpt_dir = Path(CFG['output_dir']) / 'checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)
t_start = time.time()

history = []

for epoch in range(1, CFG['epochs'] + 1):
    t0 = time.time()

    train_loss = train_epoch(model, train_loader, optimizer, criterion, scaler, epoch)
    val_metrics = validate_epoch(model, val_loader, criterion)
    scheduler.step()

    if IS_MAIN:
        epoch_time = time.time() - t0
        elapsed = time.time() - t_start
        vram = torch.cuda.max_memory_allocated(DEVICE) / 1024**3
        torch.cuda.reset_peak_memory_stats(DEVICE)
        lr_now = optimizer.param_groups[0]['lr']

        print(f'\n── Epoch {epoch:3d}/{CFG["epochs"]} ──────────────────────────')
        print(f'  Train Loss: {train_loss:.4f}  |  Val Loss: {val_metrics["loss"]:.4f}')
        print(f'  Val Macro AUC: {val_metrics["macro_auc"]:.4f}  |  LR: {lr_now:.2e}')
        print(f'  Time: {epoch_time:.0f}s epoch | {elapsed/60:.0f}min total | VRAM: {vram:.1f}GB')

        print_validation_summary(val_metrics)

        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_metrics['loss'],
            'macro_auc': val_metrics['macro_auc'],
        })

        # ── Checkpoint ─────────────────────────────────────
        current_auc = val_metrics['macro_auc']

        if current_auc > best_auc + 0.0005:
            best_auc = current_auc
            best_epoch = epoch
            patience = 0
            # DataParallel stores the actual model in .module
            state = model.module.state_dict() if N_GPUS > 1 else model.state_dict()
            ckpt_path = ckpt_dir / 'best_model.pt'
            torch.save({'epoch': epoch, 'model': state, 'auc': best_auc, 'config': CFG}, ckpt_path)
            print(f'  >> Best model saved (AUC={best_auc:.4f})')
            save_validation_report(val_metrics, CFG['output_dir'], epoch=epoch, is_best=True)
        else:
            patience += 1
            if patience >= CFG['early_stop_patience']:
                print(f'\n  Early stopping triggered at epoch {epoch}')
                break

# ── Final Report ────────────────────────────────────────────
if IS_MAIN:
    total_time = time.time() - t_start
    print(f'\n{"="*60}')
    print(f'Training Complete')
    print(f'  Best Val Macro AUC: {best_auc:.4f} (epoch {best_epoch})')
    print(f'  Total Time:         {total_time/3600:.1f} hours')
    print(f'  Early stop:         {"Yes" if patience >= CFG["early_stop_patience"] else f"No (ran all {CFG["epochs"]} epochs)"}')
    print(f'{"="*60}')

    history_df = pd.DataFrame(history)
    history_df.to_csv(Path(CFG['output_dir']) / 'training_history.csv', index=False)
    print(f'\nTraining history saved: training_history.csv')
    print(f'Best checkpoint:       checkpoints/best_model.pt')
    print(f'Validation report:     validation_report_best.csv')
    print(f'Raw predictions:       validation_predictions_best.csv')
    print(f'\nAll output files are in: {CFG["output_dir"]}')
    print(f'{"="*60}')